# Posture Coach (notebook)

Same coach as `posture_detector.py` (they share `posture_core.py`), plus tuning and history.

The camera opens in its **own window** (OpenCV can't stream live video inside a notebook). Press `q` in that window to stop and return here.

1. Run the first code cell (optional tuning)
2. Run the second cell to start coaching
3. Run the last cell to see your history

In [ ]:
import sys, os, importlib
sys.path.insert(0, os.getcwd())
import posture_core as core
import posture_detector as app
importlib.reload(core); importlib.reload(app)

# --- Optional tuning (defaults are fine to start) ---------------------------
# Dead zone = how far you can move before it cares (bigger = more forgiving).
# core.RULES['height']['dead'] = 0.10      # slumping tolerance
# core.RULES['offset']['dead'] = 0.10      # sideways head drift tolerance
# core.BREAK_EVERY_MIN = 45                # stand-up reminder interval
# core.BAD_GRACE_S = 15                    # seconds of bad posture before the chime
print('Ready. Dead zones:', {k: v['dead'] for k, v in core.RULES.items()})

In [ ]:
# Opens the coach window. Keys: r redo setup | s skip | p pause | b snooze break | d debug | q quit
app.run()

## History

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists(app.SESSIONS_CSV):
    print('No sessions yet. Run the coach for a few minutes first.')
else:
    s = pd.read_csv(app.SESSIONS_CSV)
    display(s.tail(10))
    label = s['date'] + ' ' + s['start_time']
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(label, s['avg_score'], marker='o', color='seagreen')
    ax[0].set_ylim(0, 100); ax[0].set_title('Average posture score per session')
    ax[0].tick_params(axis='x', rotation=45)
    ax[1].bar(label, s['pct_good'], color='seagreen', label='% good')
    ax[1].bar(label, s['pct_poor'], bottom=s['pct_good'], color='tomato', label='% poor')
    ax[1].set_title('Time in good vs poor posture'); ax[1].legend()
    ax[1].tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.show()

if os.path.exists(app.EPISODES_CSV):
    e = pd.read_csv(app.EPISODES_CSV)
    print('\nWhat you slouch with most (episodes of 10s+ of poor posture):')
    display(e['issue'].value_counts())